# 🎯 DTLN Model Training & Evaluation Pipeline

**⚡ GPU Runtime automatically selected** (T4 GPU)

This notebook provides an automated pipeline for:
- 📦 Installing dependencies
- 📥 Cloning dataset from GitHub (optional)
- 🏋️ Training DTLN model
- 📊 Evaluating trained model
- 🔄 Converting model to ONNX/TFLite format

**Supported Audio Formats:** WAV, MP3, FLAC, OGG, M4A, AAC, WMA, AIFF, APE

Dataset structure expected:
```
dataset/
├── train/
│   ├── clean/dataset_name/*.{wav,mp3,flac,ogg,m4a,...}
│   └── noise/dataset_name/*.{wav,mp3,flac,ogg,m4a,...}
├── validation/
│   ├── clean/dataset_name/*.{wav,mp3,flac,ogg,m4a,...}
│   └── noise/dataset_name/*.{wav,mp3,flac,ogg,m4a,...}
└── test/
    ├── clean/dataset_name/*.{wav,mp3,flac,ogg,m4a,...}
    └── noise/dataset_name/*.{wav,mp3,flac,ogg,m4a,...}
```

**Note:** All audio files will be automatically converted to 16kHz mono WAV format during training preparation.

In [ ]:
#@title **📦 Install Dependencies & Clone Repository** { display-mode: "form" }

import subprocess
import sys
import os
from pathlib import Path

# Check if GPU is available
print("🔍 Checking GPU availability...")
print("=" * 60)
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"✅ GPU detected: {len(gpus)} GPU(s) available")
        for gpu in gpus:
            print(f"   • {gpu}")
    else:
        print("⚠️ WARNING: No GPU detected!")
        print("💡 This notebook is configured for GPU runtime")
        print("💡 If you see this message, the runtime may not have started correctly")
except:
    print("⚠️ TensorFlow not yet installed, GPU check will be done after installation")

print("\n🔧 Installing system and Python dependencies...")
print("=" * 60)

try:
    # Install system dependencies for audio format support
    print("📥 Installing system audio libraries (ffmpeg, libsndfile1)...")
    subprocess.run(
        ['apt-get', 'update', '-qq'],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    subprocess.run(
        ['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1'],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    print("✅ System audio libraries installed")
    
except subprocess.CalledProcessError as e:
    print(f"❌ Failed to install system packages: {e}")
    sys.exit(1)

# Clone repository if not exists
repo_dir = Path('/content/DTLN-Retrain')

if not repo_dir.exists():
    print("\n📥 Cloning DTLN-Retrain repository...")
    try:
        subprocess.run(
            ['git', 'clone', 'https://github.com/MochNad/DTLN-Retrain.git'],
            cwd='/content',
            check=True,
            capture_output=True
        )
        print("✅ Repository cloned successfully")
    except subprocess.CalledProcessError as e:
        print("⚠️ Failed to clone repository")
        print("💡 Creating directory structure manually...")
        repo_dir.mkdir(parents=True, exist_ok=True)
        print("❌ Please manually upload DTLN files or check the repository URL")
        sys.exit(1)
else:
    print("\n✅ Repository already exists")

# Change to repository directory
os.chdir(repo_dir)
print(f"📁 Working directory: {os.getcwd()}")

# Check if requirements.txt exists
requirements_file = repo_dir / 'requirements.txt'

if not requirements_file.exists():
    print("\n⚠️ requirements.txt not found, creating compatible version...")
    # Create requirements.txt with compatible TensorFlow version for Colab
    requirements_content = """# TensorFlow and Deep Learning (compatible with Colab)
tensorflow>=2.16.0,<2.21.0

# Audio Processing
soundfile>=0.10.3
librosa>=0.9.0
wavinfo>=1.0.0
pydub>=0.25.1

# Model Conversion
tf2onnx>=1.13.0
onnx>=1.12.0

# Utilities
tqdm>=4.62.0
numpy>=1.21.0,<2.0.0
scipy>=1.7.0

# Optional: For audio resampling
resampy>=0.3.1
"""
    with open(requirements_file, 'w') as f:
        f.write(requirements_content)
    print("✅ Created requirements.txt with TensorFlow 2.16+ (compatible with Colab)")

# Install Python packages from requirements.txt
print("\n📥 Installing Python packages from requirements.txt...")
print("=" * 60)

try:
    # Use pip install with requirements.txt
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
        check=True,
        capture_output=True,
        text=True
    )
    print("✅ All Python packages installed successfully")
    
except subprocess.CalledProcessError as e:
    print(f"❌ Failed to install Python packages")
    print(f"Error: {e.stderr}")
    print("\n💡 Trying to install packages individually...")
    
    # Fallback: try installing line by line
    with open(requirements_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                try:
                    package = line.split('#')[0].strip()
                    if package:
                        print(f"   Installing {package}...")
                        subprocess.run(
                            [sys.executable, '-m', 'pip', 'install', '-q', package],
                            check=True,
                            stdout=subprocess.DEVNULL
                        )
                except:
                    print(f"   ⚠️ Failed to install {package}, continuing...")
    
    print("✅ Package installation completed with some warnings")

# Final GPU check after TensorFlow installation
print("\n🔍 Final GPU check after installation...")
print("=" * 60)
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"✅ GPU is ready: {len(gpus)} GPU(s) available")
        print(f"   TensorFlow version: {tf.__version__}")
        for gpu in gpus:
            print(f"   • {gpu.name}")
        # Enable memory growth to prevent OOM errors
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU memory growth enabled")
    else:
        print("⚠️ WARNING: No GPU detected!")
        print("💡 Training will be extremely slow without GPU")
except Exception as e:
    print(f"⚠️ Could not verify GPU: {e}")

print("\n✅ Environment setup completed!")
print("=" * 60)

In [ ]:
#@title **🛠️ Helper Functions** { display-mode: "form" }

import os
import subprocess
import sys
from pathlib import Path
from typing import List, Optional, Tuple, Dict
import json
import librosa
import soundfile as sf
from tqdm import tqdm
import shutil

# Supported audio extensions
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac', '.wma', '.aiff', '.ape', '.opus', '.webm'}

def find_datasets(base_path: Path, data_type: str) -> List[str]:
    """
    Find available datasets in the specified path.
    
    Args:
        base_path: Base path to search (e.g., /content/dataset)
        data_type: 'clean' or 'noise'
        
    Returns:
        List of dataset names
    """
    datasets = set()
    
    for split in ['train', 'validation', 'test']:
        split_path = base_path / split / data_type
        if split_path.exists():
            for item in split_path.iterdir():
                if item.is_dir():
                    datasets.add(item.name)
    
    return sorted(list(datasets))


def find_all_datasets(base_path: Path) -> Dict[str, List[str]]:
    """
    Find all available datasets (both clean and noise).
    
    Args:
        base_path: Base path to search (e.g., /content/dataset)
        
    Returns:
        Dictionary with 'clean' and 'noise' keys containing lists of dataset names
    """
    return {
        'clean': find_datasets(base_path, 'clean'),
        'noise': find_datasets(base_path, 'noise')
    }


def count_audio_files(directory: Path) -> int:
    """
    Count audio files in directory.
    
    Args:
        directory: Directory to scan
        
    Returns:
        Number of audio files
    """
    count = 0
    for ext in AUDIO_EXTENSIONS:
        count += len(list(directory.rglob(f'*{ext}')))
    return count


def convert_audio_to_wav(input_path: Path, output_path: Path, target_sr: int = 16000) -> bool:
    """
    Convert any audio format to 16kHz mono WAV format.
    
    Args:
        input_path: Input audio file path
        output_path: Output WAV file path
        target_sr: Target sampling rate (default: 16000)
        
    Returns:
        True if successful
    """
    try:
        # Load audio with librosa (handles multiple formats via ffmpeg)
        audio, sr = librosa.load(input_path, sr=target_sr, mono=True)
        
        # Save as WAV
        output_path.parent.mkdir(parents=True, exist_ok=True)
        sf.write(output_path, audio, target_sr)
        
        return True
    except Exception as e:
        print(f"⚠️ Failed to convert {input_path.name}: {e}")
        return False


def prepare_all_datasets_for_training(base_path: Path, output_path: Path,
                                      target_sr: int = 16000) -> Tuple[bool, dict, dict]:
    """
    Prepare ALL datasets by converting all audio files to 16kHz mono WAV format.
    Automatically detects all clean and noise datasets.
    
    Args:
        base_path: Base dataset path
        output_path: Output path for converted dataset
        target_sr: Target sampling rate
        
    Returns:
        Tuple of (success, stats_dict, dataset_names_dict)
    """
    print("\n🔄 Preparing ALL datasets for training...")
    print("📝 Auto-detecting datasets and converting to 16kHz mono WAV format")
    print("=" * 60)
    
    # Discover all datasets
    print("\n🔍 Scanning for available datasets...")
    all_datasets = find_all_datasets(base_path)
    
    clean_datasets = all_datasets['clean']
    noise_datasets = all_datasets['noise']
    
    print(f"📋 Found {len(clean_datasets)} clean dataset(s): {', '.join(clean_datasets) if clean_datasets else 'None'}")
    print(f"📋 Found {len(noise_datasets)} noise dataset(s): {', '.join(noise_datasets) if noise_datasets else 'None'}")
    
    if not clean_datasets and not noise_datasets:
        print("\n❌ No datasets found!")
        return False, {}, {}
    
    stats = {}
    
    for split in ['train', 'validation', 'test']:
        print(f"\n📁 Processing {split} split...")
        stats[split] = {'clean': {}, 'noise': {}}
        
        # Process all clean datasets
        if clean_datasets:
            print(f"   🧹 Converting clean datasets...")
            for dataset_name in clean_datasets:
                clean_source = base_path / split / 'clean' / dataset_name
                clean_target = output_path / split / 'clean' / dataset_name
                
                if clean_source.exists():
                    audio_files = []
                    for ext in AUDIO_EXTENSIONS:
                        audio_files.extend(list(clean_source.rglob(f'*{ext}')))
                    
                    converted_count = 0
                    for audio_file in tqdm(audio_files, desc=f"   {dataset_name}"):
                        # Preserve directory structure
                        rel_path = audio_file.relative_to(clean_source)
                        output_file = clean_target / rel_path.with_suffix('.wav')
                        
                        if convert_audio_to_wav(audio_file, output_file, target_sr):
                            converted_count += 1
                    
                    stats[split]['clean'][dataset_name] = converted_count
                    print(f"      ✅ {dataset_name}: {converted_count} files")
        
        # Process all noise datasets
        if noise_datasets:
            print(f"   🔊 Converting noise datasets...")
            for dataset_name in noise_datasets:
                noise_source = base_path / split / 'noise' / dataset_name
                noise_target = output_path / split / 'noise' / dataset_name
                
                if noise_source.exists():
                    audio_files = []
                    for ext in AUDIO_EXTENSIONS:
                        audio_files.extend(list(noise_source.rglob(f'*{ext}')))
                    
                    converted_count = 0
                    for audio_file in tqdm(audio_files, desc=f"   {dataset_name}"):
                        # Preserve directory structure
                        rel_path = audio_file.relative_to(noise_source)
                        output_file = noise_target / rel_path.with_suffix('.wav')
                        
                        if convert_audio_to_wav(audio_file, output_file, target_sr):
                            converted_count += 1
                    
                    stats[split]['noise'][dataset_name] = converted_count
                    print(f"      ✅ {dataset_name}: {converted_count} files")
    
    print("\n✅ Dataset preparation completed!")
    print("\n📊 Conversion statistics:")
    for split, type_stats in stats.items():
        print(f"   {split.capitalize()}:")
        if type_stats['clean']:
            print(f"      Clean datasets:")
            for dataset_name, count in type_stats['clean'].items():
                print(f"         • {dataset_name}: {count} files")
        if type_stats['noise']:
            print(f"      Noise datasets:")
            for dataset_name, count in type_stats['noise'].items():
                print(f"         • {dataset_name}: {count} files")
    
    dataset_names = {
        'clean': clean_datasets,
        'noise': noise_datasets
    }
    
    return True, stats, dataset_names


def prepare_dataset_for_training(base_path: Path, clean_dataset: str, 
                                 noise_dataset: str, output_path: Path,
                                 target_sr: int = 16000) -> Tuple[bool, dict]:
    """
    Prepare dataset by converting all audio files to 16kHz mono WAV format.
    
    Args:
        base_path: Base dataset path
        clean_dataset: Clean dataset name
        noise_dataset: Noise dataset name
        output_path: Output path for converted dataset
        target_sr: Target sampling rate
        
    Returns:
        Tuple of (success, stats_dict)
    """
    print("\n🔄 Preparing dataset for training...")
    print("📝 Converting all audio files to 16kHz mono WAV format")
    print("=" * 60)
    
    stats = {
        'train': {'clean': 0, 'noise': 0},
        'validation': {'clean': 0, 'noise': 0},
        'test': {'clean': 0, 'noise': 0}
    }
    
    for split in ['train', 'validation', 'test']:
        print(f"\n📁 Processing {split} split...")
        
        # Process clean files
        clean_source = base_path / split / 'clean' / clean_dataset
        clean_target = output_path / split / 'clean' / clean_dataset
        
        if clean_source.exists():
            print(f"   🧹 Converting clean audio files...")
            audio_files = []
            for ext in AUDIO_EXTENSIONS:
                audio_files.extend(list(clean_source.rglob(f'*{ext}')))
            
            for audio_file in tqdm(audio_files, desc=f"   Clean {split}"):
                # Preserve directory structure
                rel_path = audio_file.relative_to(clean_source)
                output_file = clean_target / rel_path.with_suffix('.wav')
                
                if convert_audio_to_wav(audio_file, output_file, target_sr):
                    stats[split]['clean'] += 1
        
        # Process noise files
        noise_source = base_path / split / 'noise' / noise_dataset
        noise_target = output_path / split / 'noise' / noise_dataset
        
        if noise_source.exists():
            print(f"   🔊 Converting noise audio files...")
            audio_files = []
            for ext in AUDIO_EXTENSIONS:
                audio_files.extend(list(noise_source.rglob(f'*{ext}')))
            
            for audio_file in tqdm(audio_files, desc=f"   Noise {split}"):
                # Preserve directory structure
                rel_path = audio_file.relative_to(noise_source)
                output_file = noise_target / rel_path.with_suffix('.wav')
                
                if convert_audio_to_wav(audio_file, output_file, target_sr):
                    stats[split]['noise'] += 1
    
    print("\n✅ Dataset preparation completed!")
    print("\n📊 Conversion statistics:")
    for split, counts in stats.items():
        print(f"   {split.capitalize()}:")
        print(f"      Clean: {counts['clean']} files")
        print(f"      Noise: {counts['noise']} files")
    
    return True, stats


def validate_dataset_structure(base_path: Path, clean_dataset: str, 
                               noise_dataset: str) -> Tuple[bool, str, dict]:
    """
    Validate that the dataset structure is correct.
    
    Args:
        base_path: Base dataset path
        clean_dataset: Clean dataset name
        noise_dataset: Noise dataset name
        
    Returns:
        Tuple of (is_valid, error_message, file_counts)
    """
    required_splits = ['train', 'validation']
    file_counts = {}
    
    for split in required_splits:
        clean_path = base_path / split / 'clean' / clean_dataset
        noise_path = base_path / split / 'noise' / noise_dataset
        
        if not clean_path.exists():
            return False, f"Missing: {clean_path}", {}
        
        if not noise_path.exists():
            return False, f"Missing: {noise_path}", {}
        
        # Count audio files of any supported format
        clean_count = count_audio_files(clean_path)
        noise_count = count_audio_files(noise_path)
        
        if clean_count == 0:
            return False, f"No audio files in {clean_path}", {}
        
        if noise_count == 0:
            return False, f"No audio files in {noise_path}", {}
        
        file_counts[split] = {'clean': clean_count, 'noise': noise_count}
    
    return True, "", file_counts


def prepare_mixed_dataset(base_path: Path, clean_dataset: str, 
                         noise_dataset: str, output_path: Path) -> bool:
    """
    Create mixed dataset by combining clean speech with noise.
    Simple implementation: uses noise files directly as "noisy" versions.
    For actual mixing, you would need to implement SNR-based mixing.
    
    Args:
        base_path: Base dataset path
        clean_dataset: Clean dataset name  
        noise_dataset: Noise dataset name
        output_path: Output path for mixed dataset
        
    Returns:
        True if successful
    """
    print("\n🔀 Preparing mixed dataset...")
    print("💡 Using noise files as noisy versions (simplified)")
    print("=" * 60)
    
    for split in ['train', 'validation']:
        noise_path = base_path / split / 'noise' / noise_dataset
        output_split_path = output_path / split
        output_split_path.mkdir(parents=True, exist_ok=True)
        
        # Create symlink or copy noise files
        if not (output_split_path / noise_dataset).exists():
            try:
                # Try symlink first (faster)
                (output_split_path / noise_dataset).symlink_to(
                    noise_path, target_is_directory=True
                )
                print(f"✅ Linked {split} noisy data")
            except:
                # Fallback to copy
                import shutil
                shutil.copytree(noise_path, output_split_path / noise_dataset)
                print(f"✅ Copied {split} noisy data")
    
    print("✅ Mixed dataset prepared")
    return True


def run_training(train_mix_path: str, train_speech_path: str,
                val_mix_path: str, val_speech_path: str,
                run_name: str, gpu: str = '0') -> bool:
    """
    Run DTLN training.
    
    Returns:
        True if successful
    """
    print("\n🏋️ Starting DTLN Training...")
    print("=" * 60)
    
    cmd = [
        sys.executable, 'run_training.py',
        '--train_mix', train_mix_path,
        '--train_speech', train_speech_path,
        '--val_mix', val_mix_path,
        '--val_speech', val_speech_path,
        '--run_name', run_name,
        '--gpu', gpu
    ]
    
    print(f"🔧 Command: {' '.join(cmd)}")
    print("")
    
    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True,
            bufsize=1
        )
        
        for line in iter(process.stdout.readline, ''):
            if line:
                print(line.rstrip())
                sys.stdout.flush()
        
        process.wait()
        
        if process.returncode == 0:
            print("\n✅ Training completed successfully!")
            return True
        else:
            print(f"\n❌ Training failed with code {process.returncode}")
            return False
            
    except Exception as e:
        print(f"\n❌ Training error: {e}")
        return False


def run_evaluation(input_folder: str, output_folder: str, 
                  model_path: str) -> bool:
    """
    Run DTLN evaluation on test set.
    
    Returns:
        True if successful
    """
    print("\n📊 Starting DTLN Evaluation...")
    print("=" * 60)
    
    cmd = [
        sys.executable, 'run_evaluation.py',
        '--in_folder', input_folder,
        '--out_folder', output_folder,
        '--model', model_path
    ]
    
    print(f"🔧 Command: {' '.join(cmd)}")
    print("")
    
    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True,
            bufsize=1
        )
        
        for line in iter(process.stdout.readline, ''):
            if line:
                print(line.rstrip())
                sys.stdout.flush()
        
        process.wait()
        
        if process.returncode == 0:
            print("\n✅ Evaluation completed successfully!")
            return True
        else:
            print(f"\n❌ Evaluation failed with code {process.returncode}")
            return False
            
    except Exception as e:
        print(f"\n❌ Evaluation error: {e}")
        return False


def run_conversion(weights_file: str, target_name: str, 
                  format: str = 'onnx') -> bool:
    """
    Convert trained model to ONNX or TFLite format.
    
    Args:
        weights_file: Path to .weights.h5 weights
        target_name: Target name (without extension)
        format: 'onnx' or 'tflite'
        
    Returns:
        True if successful
    """
    print(f"\n🔄 Converting model to {format.upper()}...")
    print("=" * 60)
    
    if format == 'onnx':
        cmd = [
            sys.executable, 'convert_weights_to_onnx.py',
            '--weights_file', weights_file,
            '--target_folder', target_name
        ]
    else:
        print("❌ TFLite conversion not yet implemented in this notebook")
        print("💡 Use DTLN_model.create_tf_lite_model() method directly")
        return False
    
    print(f"🔧 Command: {' '.join(cmd)}")
    print("")
    
    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True,
            bufsize=1
        )
        
        for line in iter(process.stdout.readline, ''):
            if line:
                print(line.rstrip())
                sys.stdout.flush()
        
        process.wait()
        
        if process.returncode == 0:
            print(f"\n✅ Conversion to {format.upper()} completed!")
            return True
        else:
            print(f"\n❌ Conversion failed with code {process.returncode}")
            return False
            
    except Exception as e:
        print(f"\n❌ Conversion error: {e}")
        return False


print("✅ All helper functions loaded successfully!")
print(f"📋 Supported audio formats: {', '.join(sorted(AUDIO_EXTENSIONS))}")

In [ ]:
#@title **🚀 Configure & Execute Pipeline** { display-mode: "form" }

#@markdown ---
#@markdown ### 📁 Dataset Configuration

dataset_base_path = "/content/dataset" #@param {type:"string"}

#@markdown #### Option 1: Use specific datasets (manual selection)
#@markdown Leave empty to auto-detect and convert ALL datasets:
clean_dataset_name = "" #@param {type:"string"}
noise_dataset_name = "" #@param {type:"string"}

#@markdown #### Option 2: Clone dataset from GitHub (leave empty to skip)
#@markdown Clone clean dataset from GitHub repository:
github_clean_dataset_url = "" #@param {type:"string"}

#@markdown Clone noise dataset from GitHub repository:
github_noise_dataset_url = "" #@param {type:"string"}

#@markdown ---
#@markdown ### ⚙️ Pipeline Stages

clone_dataset = False #@param {type:"boolean"}
#@markdown Clone dataset from GitHub before training

convert_to_wav = True #@param {type:"boolean"}
#@markdown Convert all audio formats to 16kHz mono WAV (independent stage)

run_training_stage = True #@param {type:"boolean"}
run_evaluation_stage = False #@param {type:"boolean"}
run_conversion_stage = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### 🎵 Audio Processing Options

target_sampling_rate = 16000 #@param {type:"integer"}
#@markdown Target sampling rate for audio conversion (Hz)

skip_existing_converted = True #@param {type:"boolean"}
#@markdown Skip conversion if converted dataset already exists

#@markdown ---
#@markdown ### 🏋️ Training Configuration

training_run_name = "DTLN_model" #@param {type:"string"}
gpu_device = "0" #@param {type:"string"}

#@markdown ---
#@markdown ### 📊 Evaluation Configuration

#@markdown Path to test dataset (will use noise dataset from test split)
evaluation_output_folder = "/content/enhanced_audio" #@param {type:"string"}

#@markdown Path to trained model weights (.weights.h5 file)
model_weights_path = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 🔄 Conversion Configuration

conversion_format = "onnx" #@param ["onnx", "tflite"]
conversion_target_name = "/content/dtln_converted" #@param {type:"string"}

#@markdown ---

# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("=" * 60)
print("🎯 DTLN PIPELINE EXECUTION")
print("=" * 60)

base_path = Path(dataset_base_path)

# ============================================================================
# STAGE -1: CLONE DATASET FROM GITHUB (if enabled)
# ============================================================================

if clone_dataset:
    print("\n" + "=" * 60)
    print("📥 STAGE -1: CLONING DATASET FROM GITHUB")
    print("=" * 60)
    
    # Clone clean dataset
    if github_clean_dataset_url:
        print("\n📥 Cloning clean dataset...")
        clean_clone_path = base_path / 'temp_clean'
        
        try:
            subprocess.run(
                ['git', 'clone', github_clean_dataset_url, str(clean_clone_path)],
                check=True,
                capture_output=True
            )
            print("✅ Clean dataset cloned successfully")
            
            # Move files to proper structure
            print("📁 Organizing clean dataset...")
            for split in ['train', 'validation', 'test']:
                source_split = clean_clone_path / split
                if source_split.exists():
                    # Find dataset name (first subfolder)
                    datasets = [d for d in source_split.iterdir() if d.is_dir()]
                    if datasets:
                        for dataset_dir in datasets:
                            target_split = base_path / split / 'clean' / dataset_dir.name
                            target_split.parent.mkdir(parents=True, exist_ok=True)
                            shutil.copytree(dataset_dir, target_split, dirs_exist_ok=True)
                            if not clean_dataset_name:
                                clean_dataset_name = dataset_dir.name
                            print(f"   ✅ {split}/clean/{dataset_dir.name}")
            
            # Cleanup
            shutil.rmtree(clean_clone_path)
            print("✅ Clean dataset organized")
            
        except subprocess.CalledProcessError as e:
            print(f"❌ Failed to clone clean dataset: {e}")
            print("💡 Please check the repository URL")
    
    # Clone noise dataset
    if github_noise_dataset_url:
        print("\n📥 Cloning noise dataset...")
        noise_clone_path = base_path / 'temp_noise'
        
        try:
            subprocess.run(
                ['git', 'clone', github_noise_dataset_url, str(noise_clone_path)],
                check=True,
                capture_output=True
            )
            print("✅ Noise dataset cloned successfully")
            
            # Move files to proper structure
            print("📁 Organizing noise dataset...")
            for split in ['train', 'validation', 'test']:
                source_split = noise_clone_path / split
                if source_split.exists():
                    # Find dataset name (first subfolder)
                    datasets = [d for d in source_split.iterdir() if d.is_dir()]
                    if datasets:
                        for dataset_dir in datasets:
                            target_split = base_path / split / 'noise' / dataset_dir.name
                            target_split.parent.mkdir(parents=True, exist_ok=True)
                            shutil.copytree(dataset_dir, target_split, dirs_exist_ok=True)
                            if not noise_dataset_name:
                                noise_dataset_name = dataset_dir.name
                            print(f"   ✅ {split}/noise/{dataset_dir.name}")
            
            # Cleanup
            shutil.rmtree(noise_clone_path)
            print("✅ Noise dataset organized")
            
        except subprocess.CalledProcessError as e:
            print(f"❌ Failed to clone noise dataset: {e}")
            print("💡 Please check the repository URL")
    
    if clean_dataset_name and noise_dataset_name:
        print(f"\n✅ STAGE -1 COMPLETED: Datasets cloned and organized")
        print(f"   Clean dataset: {clean_dataset_name}")
        print(f"   Noise dataset: {noise_dataset_name}")
    else:
        print("\n⚠️ Warning: Dataset cloning incomplete")
else:
    print("\n⏭️ STAGE -1: SKIPPED (clone_dataset = False)")

# Validate base dataset path exists
if not base_path.exists():
    if clone_dataset:
        print(f"\n⚠️ Dataset path was created but is empty")
    else:
        print(f"\n❌ Error: Dataset path does not exist: {base_path}")
        print("💡 Please run the dataset pipeline first or enable dataset cloning")
        sys.exit(1)

# ============================================================================
# STAGE 0: AUDIO CONVERSION
# ============================================================================

converted_base_path = Path('/content/dataset_converted')
selected_clean_dataset = clean_dataset_name
selected_noise_dataset = noise_dataset_name

if convert_to_wav:
    print("\n" + "=" * 60)
    print("🎵 STAGE 0: AUDIO FORMAT CONVERSION")
    print("=" * 60)
    
    # Check if converted dataset already exists
    if converted_base_path.exists() and skip_existing_converted:
        print(f"\n🔍 Checking for existing converted dataset...")
        
        # Check if there are already converted files
        existing_clean = find_datasets(converted_base_path, 'clean')
        existing_noise = find_datasets(converted_base_path, 'noise')
        
        if existing_clean or existing_noise:
            print(f"✅ Found existing converted dataset!")
            print(f"   Clean datasets: {', '.join(existing_clean) if existing_clean else 'None'}")
            print(f"   Noise datasets: {', '.join(existing_noise) if existing_noise else 'None'}")
            print(f"\n💡 Skipping conversion (using existing converted dataset)")
            print(f"💡 Set 'skip_existing_converted = False' to force reconversion")
            
            # Select datasets for training
            if not selected_clean_dataset and existing_clean:
                selected_clean_dataset = existing_clean[0]
            if not selected_noise_dataset and existing_noise:
                selected_noise_dataset = existing_noise[0]
            
            if selected_clean_dataset and selected_noise_dataset:
                print(f"\n📋 Using datasets for next stages:")
                print(f"   Clean: {selected_clean_dataset}")
                print(f"   Noise: {selected_noise_dataset}")
            
            print(f"\n✅ STAGE 0 COMPLETED: Using existing converted dataset")
        else:
            print(f"⚠️ Converted path exists but no datasets found, proceeding with conversion...")
            skip_existing_converted = False
    
    # Perform conversion if needed
    if not skip_existing_converted or not converted_base_path.exists():
        # Check if user specified datasets or wants to convert all
        use_all_datasets = not clean_dataset_name or not noise_dataset_name
        
        if use_all_datasets:
            print("\n💡 No specific datasets selected - converting ALL available datasets")
            
            # Convert all datasets
            success, stats, dataset_names = prepare_all_datasets_for_training(
                base_path=base_path,
                output_path=converted_base_path,
                target_sr=target_sampling_rate
            )
            
            if not success:
                print("\n❌ Audio conversion failed!")
                sys.exit(1)
            
            # Select first available datasets for training
            if dataset_names['clean']:
                selected_clean_dataset = dataset_names['clean'][0]
                print(f"\n💡 Selected clean dataset for training: {selected_clean_dataset}")
            
            if dataset_names['noise']:
                selected_noise_dataset = dataset_names['noise'][0]
                print(f"💡 Selected noise dataset for training: {selected_noise_dataset}")
            
            if not selected_clean_dataset or not selected_noise_dataset:
                print("\n❌ No datasets available for training after conversion!")
                sys.exit(1)
            
        else:
            print(f"\n💡 Converting specific datasets: {clean_dataset_name} (clean), {noise_dataset_name} (noise)")
            
            # Check available datasets
            print("\n🔍 Scanning for available datasets...")
            clean_datasets = find_datasets(base_path, 'clean')
            noise_datasets = find_datasets(base_path, 'noise')
            
            print(f"\n📋 Available clean datasets: {', '.join(clean_datasets) if clean_datasets else 'None'}")
            print(f"📋 Available noise datasets: {', '.join(noise_datasets) if noise_datasets else 'None'}")
            
            if clean_dataset_name not in clean_datasets:
                print(f"\n❌ Clean dataset '{clean_dataset_name}' not found!")
                print(f"💡 Available options: {', '.join(clean_datasets)}")
                sys.exit(1)
            
            if noise_dataset_name not in noise_datasets:
                print(f"\n❌ Noise dataset '{noise_dataset_name}' not found!")
                print(f"💡 Available options: {', '.join(noise_datasets)}")
                sys.exit(1)
            
            # Validate dataset structure and count files
            print("\n🔍 Validating dataset structure...")
            is_valid, error_msg, file_counts = validate_dataset_structure(
                base_path, clean_dataset_name, noise_dataset_name
            )
            
            if not is_valid:
                print(f"\n❌ Dataset validation failed: {error_msg}")
                sys.exit(1)
            
            print("✅ Dataset structure is valid")
            print("\n📊 Original dataset files:")
            for split, counts in file_counts.items():
                print(f"   {split.capitalize()}:")
                print(f"      Clean: {counts['clean']} files")
                print(f"      Noise: {counts['noise']} files")
            
            # Prepare converted dataset
            success, stats = prepare_dataset_for_training(
                base_path=base_path,
                clean_dataset=clean_dataset_name,
                noise_dataset=noise_dataset_name,
                output_path=converted_base_path,
                target_sr=target_sampling_rate
            )
            
            if not success:
                print("\n❌ Audio conversion failed!")
                sys.exit(1)
            
            selected_clean_dataset = clean_dataset_name
            selected_noise_dataset = noise_dataset_name
        
        print(f"\n✅ STAGE 0 COMPLETED: Audio converted to {target_sampling_rate}Hz mono WAV")
        print(f"📁 Converted dataset location: {converted_base_path}")

else:
    print("\n⏭️ STAGE 0: SKIPPED (convert_to_wav = False)")
    # Use original dataset path if not converting
    converted_base_path = base_path

# ============================================================================
# STAGE 1: TRAINING
# ============================================================================

if run_training_stage:
    print("\n" + "=" * 60)
    print("🏋️ STAGE 1: TRAINING")
    print("=" * 60)
    
    # If datasets not already selected, try to find them
    if not selected_clean_dataset or not selected_noise_dataset:
        print("\n🔍 Scanning for available datasets...")
        clean_datasets = find_datasets(converted_base_path, 'clean')
        noise_datasets = find_datasets(converted_base_path, 'noise')
        
        if not clean_datasets or not noise_datasets:
            print("\n❌ No datasets found for training!")
            print("💡 Please enable 'convert_to_wav' or ensure datasets exist")
            sys.exit(1)
        
        # Use first available if not specified
        if not selected_clean_dataset:
            selected_clean_dataset = clean_datasets[0]
        if not selected_noise_dataset:
            selected_noise_dataset = noise_datasets[0]
        
        print(f"\n📋 Auto-selected datasets:")
        print(f"   Clean: {selected_clean_dataset}")
        print(f"   Noise: {selected_noise_dataset}")
    
    # Validate that we have dataset names
    if not selected_clean_dataset or not selected_noise_dataset:
        print("\n❌ Error: No datasets available for training!")
        print("💡 Please ensure datasets exist or enable conversion stage")
        sys.exit(1)
    
    # Prepare paths (use converted dataset if available)
    train_mix_path = str(converted_base_path / 'train' / 'noise' / selected_noise_dataset)
    train_speech_path = str(converted_base_path / 'train' / 'clean' / selected_clean_dataset)
    val_mix_path = str(converted_base_path / 'validation' / 'noise' / selected_noise_dataset)
    val_speech_path = str(converted_base_path / 'validation' / 'clean' / selected_clean_dataset)
    
    # Validate paths exist
    if not Path(train_mix_path).exists():
        print(f"\n❌ Training noise path not found: {train_mix_path}")
        sys.exit(1)
    if not Path(train_speech_path).exists():
        print(f"\n❌ Training clean path not found: {train_speech_path}")
        sys.exit(1)
    if not Path(val_mix_path).exists():
        print(f"\n❌ Validation noise path not found: {val_mix_path}")
        sys.exit(1)
    if not Path(val_speech_path).exists():
        print(f"\n❌ Validation clean path not found: {val_speech_path}")
        sys.exit(1)
    
    print(f"\n📁 Training paths:")
    print(f"   Noisy (mix): {train_mix_path}")
    print(f"   Clean: {train_speech_path}")
    print(f"   Val noisy: {val_mix_path}")
    print(f"   Val clean: {val_speech_path}")
    
    # Run training
    success = run_training(
        train_mix_path=train_mix_path,
        train_speech_path=train_speech_path,
        val_mix_path=val_mix_path,
        val_speech_path=val_speech_path,
        run_name=training_run_name,
        gpu=gpu_device
    )
    
    if success:
        model_save_path = Path(f'./models_{training_run_name}')
        weights_file = model_save_path / f'{training_run_name}.weights.h5'
        print(f"\n📦 Model saved to: {weights_file}")
        print(f"📊 Training logs: {model_save_path / f'training_{training_run_name}.log'}")
    else:
        print("\n❌ Training failed!")
        sys.exit(1)

else:
    print("\n⏭️ STAGE 1: SKIPPED (run_training_stage = False)")

# ============================================================================
# STAGE 2: EVALUATION
# ============================================================================

if run_evaluation_stage:
    print("\n" + "=" * 60)
    print("📊 STAGE 2: EVALUATION")
    print("=" * 60)
    
    # Determine model weights path
    if not model_weights_path:
        # Try to use model from training stage
        if run_training_stage:
            model_weights_path = str(Path(f'./models_{training_run_name}') / f'{training_run_name}.weights.h5')
            print(f"💡 Using model from training: {model_weights_path}")
        else:
            print("\n❌ Error: Model weights path is required!")
            print("💡 Please specify 'model_weights_path' or run training stage first")
            sys.exit(1)
    
    # Check if model exists
    if not Path(model_weights_path).exists():
        print(f"\n❌ Model weights not found: {model_weights_path}")
        sys.exit(1)
    
    # Use test set noise data as input (from converted dataset if available)
    if not selected_noise_dataset:
        print("\n❌ No noise dataset available for evaluation!")
        sys.exit(1)
    
    test_input_path = str(converted_base_path / 'test' / 'noise' / selected_noise_dataset)
    
    if not Path(test_input_path).exists():
        print(f"\n❌ Test dataset not found: {test_input_path}")
        print("💡 Make sure your dataset has a 'test' split")
        sys.exit(1)
    
    print(f"\n📁 Evaluation paths:")
    print(f"   Input (noisy): {test_input_path}")
    print(f"   Output (enhanced): {evaluation_output_folder}")
    print(f"   Model: {model_weights_path}")
    
    # Run evaluation
    success = run_evaluation(
        input_folder=test_input_path,
        output_folder=evaluation_output_folder,
        model_path=model_weights_path
    )
    
    if success:
        print(f"\n✅ Enhanced audio saved to: {evaluation_output_folder}")
    else:
        print("\n❌ Evaluation failed!")
        sys.exit(1)

else:
    print("\n⏭️ STAGE 2: SKIPPED (run_evaluation_stage = False)")

# ============================================================================
# STAGE 3: CONVERSION
# ============================================================================

if run_conversion_stage:
    print("\n" + "=" * 60)
    print("🔄 STAGE 3: MODEL CONVERSION")
    print("=" * 60)
    
    # Determine model weights path
    if not model_weights_path:
        if run_training_stage:
            model_weights_path = str(Path(f'./models_{training_run_name}') / f'{training_run_name}.weights.h5')
            print(f"💡 Using model from training: {model_weights_path}")
        else:
            print("\n❌ Error: Model weights path is required!")
            print("💡 Please specify 'model_weights_path' or run training stage first")
            sys.exit(1)
    
    # Check if model exists
    if not Path(model_weights_path).exists():
        print(f"\n❌ Model weights not found: {model_weights_path}")
        sys.exit(1)
    
    print(f"\n📁 Conversion configuration:")
    print(f"   Input model: {model_weights_path}")
    print(f"   Output format: {conversion_format.upper()}")
    print(f"   Target name: {conversion_target_name}")
    
    # Run conversion
    success = run_conversion(
        weights_file=model_weights_path,
        target_name=conversion_target_name,
        format=conversion_format
    )
    
    if success:
        if conversion_format == 'onnx':
            print(f"\n✅ ONNX models saved:")
            print(f"   • {conversion_target_name}_1.onnx")
            print(f"   • {conversion_target_name}_2.onnx")
    else:
        print("\n❌ Conversion failed!")
        sys.exit(1)

else:
    print("\n⏭️ STAGE 3: SKIPPED (run_conversion_stage = False)")

# ============================================================================
# CLEANUP
# ============================================================================

if convert_to_wav and converted_base_path != base_path:
    print("\n" + "=" * 60)
    print("🧹 CLEANUP")
    print("=" * 60)
    
    try:
        # Optionally clean up converted files after training
        cleanup_converted = False  # Set to True if you want to clean up
        
        if cleanup_converted and converted_base_path.exists():
            print("🗑️ Cleaning up converted audio files...")
            shutil.rmtree(converted_base_path)
            print("✅ Converted files cleaned up")
        else:
            print(f"💾 Converted files kept at: {converted_base_path}")
            print("💡 You can manually delete this folder if needed")
    except Exception as e:
        print(f"⚠️ Warning: Could not clean converted files: {e}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 60)
print("🎉 PIPELINE COMPLETED!")
print("=" * 60)

executed_stages = []
if clone_dataset:
    executed_stages.append("Clone Dataset")
if convert_to_wav:
    executed_stages.append("Audio Conversion")
if run_training_stage:
    executed_stages.append("Training")
if run_evaluation_stage:
    executed_stages.append("Evaluation")
if run_conversion_stage:
    executed_stages.append("Model Conversion")

print(f"\n✅ Executed stages: {', '.join(executed_stages) if executed_stages else 'None'}")

if convert_to_wav:
    print(f"\n🎵 Audio files converted to {target_sampling_rate}Hz mono WAV")
    print(f"   Location: {converted_base_path}")
    if selected_clean_dataset and selected_noise_dataset:
        print(f"   Clean dataset used: {selected_clean_dataset}")
        print(f"   Noise dataset used: {selected_noise_dataset}")

if run_training_stage:
    print(f"\n📦 Trained model: ./models_{training_run_name}/{training_run_name}.weights.h5")

if run_evaluation_stage:
    print(f"📊 Enhanced audio: {evaluation_output_folder}")

if run_conversion_stage:
    print(f"🔄 Converted model: {conversion_target_name}_{{1,2}}.{conversion_format}")

print("\n💡 Next steps:")
if clone_dataset and not convert_to_wav:
    print("   • Enable 'convert_to_wav' to prepare dataset for training")
elif convert_to_wav and not run_training_stage:
    print("   • Enable 'run_training_stage' to train the model")
elif run_training_stage and not run_evaluation_stage:
    print("   • Enable 'run_evaluation_stage' to test the model")
elif run_evaluation_stage and not run_conversion_stage:
    print("   • Enable 'run_conversion_stage' to export the model")
elif not any([clone_dataset, convert_to_wav, run_training_stage, run_evaluation_stage, run_conversion_stage]):
    print("   • Enable at least one stage to run the pipeline")
else:
    print("   • Your model is ready for deployment!")

print(f"\n📋 Supported audio formats: WAV, MP3, FLAC, OGG, M4A, AAC, WMA, AIFF, APE, OPUS, WEBM")